# DCE-MRI — patient-level evaluation

Patient-level evaluation of the completed DCE-MRI runs using the finalized model definitions, selected checkpoints, and fixed validation thresholds.


In [ ]:
from __future__ import annotations
import ast, gc, importlib.util, inspect, json, math, os, re, sys, warnings, zipfile
from collections import defaultdict
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings('ignore')
try:
    from scipy.ndimage import binary_erosion, distance_transform_edt
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working/ROI_MRI_TRACKB_FINAL_PATIENT_LEVEL_EVALUATION')
METRICS = WORK/'metrics'; LOGS = WORK/'logs'
for p in [WORK, METRICS, LOGS]: p.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'
BATCH_SIZE = 12 if USE_AMP else 2
NUM_WORKERS = 2

RUN_TRAINING = False
THRESHOLD_SEARCH_ENABLED = False
assert not RUN_TRAINING and not THRESHOLD_SEARCH_ENABLED

SPLITS = ['validation','test','external']
MODELS = ['unet','attention_unet','swin_tiny_unet']
SEEDS = [42,123,2025]
RUNS = [(m,s) for m in MODELS for s in SEEDS]

FROZEN_THRESHOLDS = {
 ('swin_tiny_unet',42):0.70, ('swin_tiny_unet',123):0.80, ('swin_tiny_unet',2025):0.70,
 ('unet',42):0.80, ('unet',123):0.75, ('unet',2025):0.75,
 ('attention_unet',42):0.80, ('attention_unet',123):0.65, ('attention_unet',2025):0.75,
}
EXPECTED_PARAMS = {'unet':7763041,'attention_unet':7851773,'swin_tiny_unet':38350819}

# Optional manual overrides when automatic discovery is insufficient.
MANIFEST_OVERRIDE = None
MODEL_SOURCE_OVERRIDE = None
CHECKPOINT_OVERRIDES = {(m,s):None for m,s in RUNS}

COMPUTE_BOUNDARY_METRICS = False
BOUNDARY_SPLITS = ['test','external']

print('Device:', DEVICE)
print('Output:', WORK)

In [ ]:
def files(suffixes):
    return sorted(p for p in INPUT.rglob('*') if p.is_file() and p.suffix.lower() in suffixes)

CSV_FILES = files(('.csv',)); NPZ_FILES = files(('.npz',)); CKPT_FILES = files(('.pt','.pth','.ckpt'))
PY_FILES = files(('.py',)); IPYNB_FILES = files(('.ipynb',))
print('CSV:',len(CSV_FILES),'NPZ:',len(NPZ_FILES),'checkpoints:',len(CKPT_FILES))

if MANIFEST_OVERRIDE:
    manifest_path = Path(MANIFEST_OVERRIDE)
else:
    preferred = ['roi_mri_manifest.csv','roi_mri_manifest_absolute_paths.csv']
    manifest_path = None
    for name in preferred:
        matches = [p for p in CSV_FILES if p.name.lower()==name]
        if matches:
            manifest_path = max(matches,key=lambda p:p.stat().st_size); break
    if manifest_path is None:
        raise FileNotFoundError('MRI manifest not found. Add ROI_MRI_Crops_256_v1.')

manifest = pd.read_csv(manifest_path)
required = {'sample_id','dataset','split','patient_id','case_id'}
if required-set(manifest.columns): raise ValueError(f'Missing manifest columns: {required-set(manifest.columns)}')

sets = {s:set(manifest.loc[manifest.split.astype(str).str.lower()==s,'patient_id'].astype(str)) for s in ['train','validation','test','external']}
leakage = {
 'train_vs_validation':len(sets['train']&sets['validation']),
 'train_vs_test':len(sets['train']&sets['test']),
 'validation_vs_test':len(sets['validation']&sets['test']),
 'development_vs_external':len((sets['train']|sets['validation']|sets['test'])&sets['external'])
}
assert all(v==0 for v in leakage.values()), leakage
(LOGS/'split_leakage_audit.json').write_text(json.dumps(leakage,indent=2))
print('Manifest:',manifest_path,'rows:',len(manifest),'leakage:',leakage)

In [ ]:
by_name = defaultdict(list)
for p in NPZ_FILES: by_name[p.name].append(p)

def resolve_npz(row):
    sample_name = f"{row['sample_id']}.npz"
    if len(by_name[sample_name])==1: return str(by_name[sample_name][0])
    for col in ['npz_path_original','npz_path']:
        if col in row and pd.notna(row[col]):
            raw = str(row[col]).replace(chr(92), '/')
            if '/npz/' in raw:
                suffix = raw.split('/npz/',1)[1]
                matches = [p for p in NPZ_FILES if str(p).replace(chr(92), '/').endswith('/npz/'+suffix)]
                if len(matches)==1: return str(matches[0])
    matches = by_name[sample_name]
    ds,sp = str(row.dataset).lower(),str(row.split).lower()
    narrowed = [p for p in matches if ds in str(p).lower() and sp in str(p).lower()]
    if len(narrowed)==1: return str(narrowed[0])
    raise FileNotFoundError(f"NPZ unresolved: {row['sample_id']}")

manifest = manifest.copy()
manifest['npz_path_resolved'] = manifest.apply(resolve_npz,axis=1)
assert manifest.npz_path_resolved.map(lambda x:Path(x).exists()).all()
manifest.to_csv(LOGS/'roi_mri_manifest_resolved.csv',index=False)

for _,r in manifest.sample(min(12,len(manifest)),random_state=2026).iterrows():
    with np.load(r.npz_path_resolved) as z:
        im,ma = z['image'],z['mask']
        assert im.shape==(256,256,3) and im.dtype==np.float32
        assert ma.shape==(256,256) and ma.dtype==np.uint8 and set(np.unique(ma))<={0,1} and ma.sum()>0
print('Resolved NPZ:',manifest.npz_path_resolved.nunique(),'structure check: PASS')

### Loader fix


## V6 — exact source extraction


## Definitive V7 — direct exact architectures


In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    from torchvision.models import swin_t
    TORCHVISION_OK = True
except Exception as exc:
    swin_t = None
    TORCHVISION_OK = False
    raise RuntimeError(
        "torchvision.models.swin_t is required to reconstruct the exact "
        f"Track B Swin checkpoint. Original import error: {exc}"
    ) from exc


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class UNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=1, base=32):
        super().__init__()
        self.e1 = ConvBlock(in_ch, base)
        self.e2 = ConvBlock(base, base * 2)
        self.e3 = ConvBlock(base * 2, base * 4)
        self.e4 = ConvBlock(base * 4, base * 8)
        self.pool = nn.MaxPool2d(2)
        self.center = ConvBlock(base * 8, base * 16)
        self.u4 = nn.ConvTranspose2d(base * 16, base * 8, 2, 2)
        self.d4 = ConvBlock(base * 16, base * 8)
        self.u3 = nn.ConvTranspose2d(base * 8, base * 4, 2, 2)
        self.d3 = ConvBlock(base * 8, base * 4)
        self.u2 = nn.ConvTranspose2d(base * 4, base * 2, 2, 2)
        self.d2 = ConvBlock(base * 4, base * 2)
        self.u1 = nn.ConvTranspose2d(base * 2, base, 2, 2)
        self.d1 = ConvBlock(base * 2, base)
        self.out = nn.Conv2d(base, out_ch, 1)

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        e4 = self.e4(self.pool(e3))
        c = self.center(self.pool(e4))
        d4 = self.d4(torch.cat([self.u4(c), e4], 1))
        d3 = self.d3(torch.cat([self.u3(d4), e3], 1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], 1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], 1))
        return self.out(d1)


class AttentionGate(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, 1, bias=True),
            nn.BatchNorm2d(F_int),
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, 1, bias=True),
            nn.BatchNorm2d(F_int),
        )
        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, 1, bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid(),
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        psi = self.relu(self.W_g(g) + self.W_x(x))
        psi = self.psi(psi)
        return x * psi


class AttentionUNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=1, base=32):
        super().__init__()
        self.e1 = ConvBlock(in_ch, base)
        self.e2 = ConvBlock(base, base * 2)
        self.e3 = ConvBlock(base * 2, base * 4)
        self.e4 = ConvBlock(base * 4, base * 8)
        self.pool = nn.MaxPool2d(2)
        self.center = ConvBlock(base * 8, base * 16)
        self.u4 = nn.ConvTranspose2d(base * 16, base * 8, 2, 2)
        self.a4 = AttentionGate(base * 8, base * 8, base * 4)
        self.d4 = ConvBlock(base * 16, base * 8)
        self.u3 = nn.ConvTranspose2d(base * 8, base * 4, 2, 2)
        self.a3 = AttentionGate(base * 4, base * 4, base * 2)
        self.d3 = ConvBlock(base * 8, base * 4)
        self.u2 = nn.ConvTranspose2d(base * 4, base * 2, 2, 2)
        self.a2 = AttentionGate(base * 2, base * 2, base)
        self.d2 = ConvBlock(base * 4, base * 2)
        self.u1 = nn.ConvTranspose2d(base * 2, base, 2, 2)
        self.a1 = AttentionGate(base, base, base // 2)
        self.d1 = ConvBlock(base * 2, base)
        self.out = nn.Conv2d(base, out_ch, 1)

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        e4 = self.e4(self.pool(e3))
        c = self.center(self.pool(e4))
        u4 = self.u4(c)
        d4 = self.d4(torch.cat([u4, self.a4(u4, e4)], 1))
        u3 = self.u3(d4)
        d3 = self.d3(torch.cat([u3, self.a3(u3, e3)], 1))
        u2 = self.u2(d3)
        d2 = self.d2(torch.cat([u2, self.a2(u2, e2)], 1))
        u1 = self.u1(d2)
        d1 = self.d1(torch.cat([u1, self.a1(u1, e1)], 1))
        return self.out(d1)


class SwinTinyUNet(nn.Module):
    def __init__(self, out_ch=1):
        super().__init__()
        if not TORCHVISION_OK or swin_t is None:
            raise RuntimeError('torchvision.models.swin_t unavailable')
        self.swin = swin_t(weights=None)
        self.features = self.swin.features
        self.center = ConvBlock(768, 512)
        self.up3 = nn.ConvTranspose2d(512, 384, 2, 2)
        self.dec3 = ConvBlock(384 + 384, 256)
        self.up2 = nn.ConvTranspose2d(256, 192, 2, 2)
        self.dec2 = ConvBlock(192 + 192, 128)
        self.up1 = nn.ConvTranspose2d(128, 96, 2, 2)
        self.dec1 = ConvBlock(96 + 96, 64)
        self.up0 = nn.ConvTranspose2d(64, 32, 2, 2)
        self.dec0 = ConvBlock(32, 32)
        self.up_final = nn.ConvTranspose2d(32, 32, 2, 2)
        self.out = nn.Conv2d(32, out_ch, 1)

    def _to_nchw(self, x):
        if x.ndim == 4 and x.shape[1] not in [96, 192, 384, 768]:
            return x.permute(0, 3, 1, 2).contiguous()
        return x

    def forward(self, x):
        feats = []
        y = x
        for i, layer in enumerate(self.features):
            y = layer(y)
            if i in [1, 3, 5, 7]:
                feats.append(self._to_nchw(y))
        if len(feats) != 4:
            raise RuntimeError(f'Expected 4 Swin features, got {len(feats)}')
        f1, f2, f3, f4 = feats
        c = self.center(f4)
        d3 = self.dec3(torch.cat([self.up3(c), f3], 1))
        d2 = self.dec2(torch.cat([self.up2(d3), f2], 1))
        d1 = self.dec1(torch.cat([self.up1(d2), f1], 1))
        d0 = self.dec0(self.up0(d1))
        out = self.out(self.up_final(d0))
        if out.shape[-2:] != x.shape[-2:]:
            out = F.interpolate(
                out,
                size=x.shape[-2:],
                mode='bilinear',
                align_corners=False,
            )
        return out


def build_model(name):
    if name == 'unet':
        return UNet(in_ch=3, out_ch=1)
    if name == 'attention_unet':
        return AttentionUNet(in_ch=3, out_ch=1)
    if name == 'swin_tiny_unet':
        return SwinTinyUNet(out_ch=1)
    raise ValueError(name)


MODEL_NS = {
    'ConvBlock': ConvBlock,
    'UNet': UNet,
    'AttentionGate': AttentionGate,
    'AttentionUNet': AttentionUNet,
    'SwinTinyUNet': SwinTinyUNet,
    'build_model': build_model,
}

EXPECTED_EXACT_COUNTS = {
    'unet': 7_763_041,
    'attention_unet': 7_851_773,
    'swin_tiny_unet': 38_350_819,
}

architecture_audit_rows = []
for architecture_name in ['unet', 'attention_unet', 'swin_tiny_unet']:
    audit_model = build_model(architecture_name)
    parameter_count = sum(p.numel() for p in audit_model.parameters())
    expected_count = EXPECTED_EXACT_COUNTS[architecture_name]
    architecture_audit_rows.append({
        'model': architecture_name,
        'parameter_count': parameter_count,
        'expected_parameter_count': expected_count,
        'exact_match': parameter_count == expected_count,
    })
    if parameter_count != expected_count:
        raise RuntimeError(
            f'Exact architecture count mismatch for {architecture_name}: '
            f'{parameter_count:,} vs {expected_count:,}'
        )
    del audit_model

architecture_audit = pd.DataFrame(architecture_audit_rows)
architecture_audit.to_csv(
    LOGS / 'exact_architecture_parameter_audit.csv',
    index=False,
)
print(architecture_audit.to_string(index=False))
print('Exact Track B architectures: PASS')


In [ ]:
def nparams(model):
    return sum(parameter.numel() for parameter in model.parameters())


def build_exact(model_name, state_dict_candidates=None):
    """
    Instantiate the exact Phase 2 architecture directly.

    There is no name guessing, source parsing or constructor sweep in V7.
    """
    if model_name not in EXPECTED_PARAMS:
        raise KeyError(f'Unknown Track B model: {model_name}')

    model = build_model(model_name)
    actual = nparams(model)
    expected = EXPECTED_PARAMS[model_name]

    if actual != expected:
        raise RuntimeError(
            f'Exact {model_name} parameter-count mismatch: '
            f'{actual:,} vs {expected:,}'
        )

    if state_dict_candidates:
        compatibility_rows = []
        for candidate_path, candidate in state_dict_candidates:
            for variant_label, variant in state_dict_variants(candidate):
                compatibility = state_dict_compatibility(model, variant)
                compatibility_rows.append({
                    'candidate_path': candidate_path,
                    'variant': variant_label,
                    'matched': compatibility['matched'],
                    'reference_keys': compatibility['reference_key_count'],
                    'candidate_keys': compatibility['candidate_key_count'],
                    'missing_count': len(compatibility['missing']),
                    'unexpected_count': len(compatibility['unexpected']),
                    'shape_mismatch_count': len(compatibility['shape_mismatches']),
                    'exact': compatibility['exact'],
                })
                if compatibility['exact']:
                    print(
                        f'{model_name}: exact architecture reconstructed; '
                        f'{actual:,} parameters; checkpoint keys/shapes match '
                        f'at {candidate_path} ({variant_label}).'
                    )
                    return model

        compatibility_frame = pd.DataFrame(compatibility_rows)
        diagnostic_path = LOGS / f'preload_compatibility_{model_name}.csv'
        compatibility_frame.to_csv(diagnostic_path, index=False)
        best = (
            compatibility_frame.sort_values(
                ['matched', 'missing_count', 'unexpected_count'],
                ascending=[False, True, True],
            ).head(10)
            if not compatibility_frame.empty
            else compatibility_frame
        )
        raise RuntimeError(
            f'The exact {model_name} architecture was instantiated, but no '
            f'checkpoint state_dict matched all keys and tensor shapes.\n'
            f'Best candidates:\n{best.to_string(index=False)}\n'
            f'Diagnostic: {diagnostic_path}'
        )

    print(
        f'{model_name}: exact architecture reconstructed; '
        f'{actual:,} parameters.'
    )
    return model


In [ ]:

def infer_model_seed(path):
    low = path.name.lower().replace('-', '_')
    model = (
        'attention_unet'
        if ('attention' in low and 'unet' in low)
        else (
            'swin_tiny_unet'
            if 'swin' in low
            else ('unet' if 'unet' in low else None)
        )
    )
    seed = next(
        (
            s for s in SEEDS
            if re.search(rf'(^|[^0-9]){s}([^0-9]|$)', low)
        ),
        None,
    )
    return model, seed

def discover_checkpoints():
    grouped = defaultdict(list)
    resolved = {}

    for key, value in CHECKPOINT_OVERRIDES.items():
        if value:
            p = Path(value)
            if not p.exists():
                raise FileNotFoundError(f'Checkpoint override not found: {p}')
            resolved[key] = p

    for p in CKPT_FILES:
        model, seed = infer_model_seed(p)
        if model and seed:
            grouped[(model, seed)].append(p)

    for key in RUNS:
        if key in resolved:
            continue
        choices = grouped[key]
        if choices:
            resolved[key] = sorted(
                choices,
                key=lambda p: (
                    2 if 'best' in p.name.lower() else 0,
                    1 if 'checkpoint' in p.name.lower() else 0,
                    p.stat().st_mtime,
                ),
                reverse=True,
            )[0]

    missing = [key for key in RUNS if key not in resolved]
    if missing:
        raise FileNotFoundError(
            f'Missing checkpoints: {missing}. '
            'Use CHECKPOINT_OVERRIDES if filenames are opaque.'
        )
    return resolved

CHECKPOINTS = discover_checkpoints()

checkpoint_table = pd.DataFrame(
    [
        {
            'model': model,
            'seed': seed,
            'threshold': FROZEN_THRESHOLDS[(model, seed)],
            'checkpoint_path': str(CHECKPOINTS[(model, seed)]),
            'size_mb': CHECKPOINTS[(model, seed)].stat().st_size / (1024 ** 2),
        }
        for model, seed in RUNS
    ]
)
checkpoint_table.to_csv(LOGS / 'checkpoints_used.csv', index=False)
print(checkpoint_table.to_string(index=False))

COMMON_PREFIXES = (
    'module.',
    '_orig_mod.',
    'model.',
    'net.',
    'network.',
    'segmentation_model.',
)

PREFERRED_STATE_KEYS = (
    'model_state_dict',
    'state_dict',
    'best_model_state_dict',
    'model_state',
    'best_model_state',
    'network_state_dict',
    'net_state_dict',
    'weights',
    'params',
    'model_weights',
    'checkpoint',
    'payload',
)

def full_model_from(obj):
    if isinstance(obj, nn.Module):
        return obj

    if isinstance(obj, dict):
        for key in (
            'model',
            'best_model',
            'net',
            'network',
            'module',
            'segmentation_model',
        ):
            value = obj.get(key)
            if isinstance(value, nn.Module):
                return value

    return None

def checkpoint_structure_summary(obj, max_depth=3, max_items=20):
    """
    Produce a compact diagnostic description without printing tensors.
    """
    lines = []

    def visit(value, path='root', depth=0):
        if depth > max_depth or len(lines) >= max_items:
            return

        if isinstance(value, nn.Module):
            lines.append(
                f'{path}: nn.Module({value.__class__.__name__}, '
                f'{nparams(value):,} params)'
            )
            return

        if isinstance(value, dict):
            keys = list(value.keys())
            shown = [str(k) for k in keys[:12]]
            lines.append(
                f'{path}: dict(len={len(value)}, keys={shown})'
            )
            ordered = []
            for key in PREFERRED_STATE_KEYS:
                if key in value:
                    ordered.append(key)
            ordered.extend(
                key for key in keys
                if key not in ordered
            )
            for key in ordered[:10]:
                child = value[key]
                if isinstance(child, (dict, list, tuple, nn.Module)):
                    visit(child, f'{path}.{key}', depth + 1)
            return

        if isinstance(value, (list, tuple)):
            lines.append(f'{path}: {type(value).__name__}(len={len(value)})')
            for i, child in enumerate(value[:5]):
                if isinstance(child, (dict, list, tuple, nn.Module)):
                    visit(child, f'{path}[{i}]', depth + 1)
            return

        lines.append(f'{path}: {type(value).__name__}')

    visit(obj)
    return lines

def _tensor_mapping_candidate(value):
    """
    Return a tensor-only mapping when value resembles a PyTorch state_dict.

    Pure state_dict mappings are accepted. Mappings containing a small amount
    of scalar metadata are also accepted by retaining only tensor entries.
    Optimizer sub-states are rejected by requiring at least eight tensors and
    string parameter-like keys.
    """
    if not isinstance(value, dict) or not value:
        return None

    tensor_items = {
        str(key): item
        for key, item in value.items()
        if torch.is_tensor(item)
    }

    if len(tensor_items) < 8:
        return None

    parameter_like = sum(
        (
            '.' in key
            or key.endswith('weight')
            or key.endswith('bias')
            or key.endswith('running_mean')
            or key.endswith('running_var')
            or key.endswith('num_batches_tracked')
        )
        for key in tensor_items
    )

    if parameter_like < max(5, int(0.50 * len(tensor_items))):
        return None

    tensor_ratio = len(tensor_items) / max(len(value), 1)
    if tensor_ratio < 0.50:
        return None

    return tensor_items

def find_state_dict_candidates(obj, max_depth=8):
    """
    Recursively discover all plausible model state_dict mappings.
    The traversal supports arbitrarily named wrappers such as:
      checkpoint -> payload -> best_model -> model_state
    """
    candidates = []
    visited = set()

    def walk(value, path='root', depth=0):
        if depth > max_depth:
            return

        object_id = id(value)
        if object_id in visited:
            return
        visited.add(object_id)

        candidate = _tensor_mapping_candidate(value)
        if candidate is not None:
            candidates.append((path, candidate))

        if isinstance(value, dict):
            ordered_keys = []
            for key in PREFERRED_STATE_KEYS:
                if key in value:
                    ordered_keys.append(key)
            ordered_keys.extend(
                key for key in value.keys()
                if key not in ordered_keys
            )

            for key in ordered_keys:
                child = value[key]
                if isinstance(child, (dict, list, tuple, nn.Module)):
                    walk(child, f'{path}.{key}', depth + 1)

        elif isinstance(value, (list, tuple)):
            for index, child in enumerate(value):
                if isinstance(child, (dict, list, tuple, nn.Module)):
                    walk(child, f'{path}[{index}]', depth + 1)

        elif hasattr(value, '__dict__'):
            attrs = vars(value)
            if isinstance(attrs, dict):
                walk(attrs, f'{path}.__dict__', depth + 1)

        if (
            not isinstance(value, nn.Module)
            and hasattr(value, 'state_dict')
            and callable(value.state_dict)
        ):
            try:
                state = value.state_dict()
                if isinstance(state, dict):
                    walk(state, f'{path}.state_dict()', depth + 1)
            except Exception:
                pass

    walk(obj)
    return candidates

def state_dict_variants(state_dict):
    """
    Generate prefix-normalized variants while preserving tensor values.
    """
    variants = []
    seen_key_sets = set()

    def add(candidate, label):
        if not candidate:
            return
        signature = tuple(candidate.keys())
        if signature not in seen_key_sets:
            seen_key_sets.add(signature)
            variants.append((label, candidate))

    current = dict(state_dict)
    add(current, 'original')

    changed = True
    iteration = 0
    while changed and current and iteration < 8:
        changed = False
        iteration += 1

        for prefix in COMMON_PREFIXES:
            if all(key.startswith(prefix) for key in current):
                current = {
                    key[len(prefix):]: value
                    for key, value in current.items()
                }
                add(current, f'stripped:{prefix}')
                changed = True
                break

        if changed:
            continue

        first_parts = {
            key.split('.', 1)[0]
            for key in current
            if '.' in key
        }
        if len(first_parts) == 1:
            root = next(iter(first_parts)) + '.'
            generic = {
                key[len(root):]: value
                for key, value in current.items()
                if key.startswith(root)
            }
            if len(generic) == len(current):
                current = generic
                add(current, f'stripped_shared_root:{root}')
                changed = True

    return variants

def state_dict_compatibility(model, candidate):
    """
    Compare key sets and tensor shapes before mutating the model.
    """
    reference = model.state_dict()
    ref_keys = set(reference.keys())
    cand_keys = set(candidate.keys())

    missing = sorted(ref_keys - cand_keys)
    unexpected = sorted(cand_keys - ref_keys)

    shape_mismatches = []
    for key in sorted(ref_keys & cand_keys):
        ref_shape = tuple(reference[key].shape)
        cand_shape = tuple(candidate[key].shape)
        if ref_shape != cand_shape:
            shape_mismatches.append((key, cand_shape, ref_shape))

    exact = (
        not missing
        and not unexpected
        and not shape_mismatches
    )
    matched = len(ref_keys & cand_keys) - len(shape_mismatches)

    return {
        'exact': exact,
        'matched': matched,
        'reference_key_count': len(ref_keys),
        'candidate_key_count': len(cand_keys),
        'missing': missing,
        'unexpected': unexpected,
        'shape_mismatches': shape_mismatches,
    }

def load_state_dict_recursively(model_name, checkpoint_object, checkpoint_path):
    candidates = find_state_dict_candidates(checkpoint_object)

    if not candidates:
        structure = '\n'.join(
            '  - ' + line
            for line in checkpoint_structure_summary(checkpoint_object)
        )
        raise RuntimeError(
            f'No tensor state_dict candidate was found in {checkpoint_path}.\n'
            f'Checkpoint structure:\n{structure}'
        )

    model = build_exact(model_name, state_dict_candidates=candidates)
    diagnostics = []

    for candidate_path, candidate in candidates:
        for variant_label, variant in state_dict_variants(candidate):
            compatibility = state_dict_compatibility(model, variant)
            diagnostics.append(
                {
                    'candidate_path': candidate_path,
                    'variant': variant_label,
                    'matched': compatibility['matched'],
                    'reference_keys': compatibility['reference_key_count'],
                    'candidate_keys': compatibility['candidate_key_count'],
                    'missing_count': len(compatibility['missing']),
                    'unexpected_count': len(compatibility['unexpected']),
                    'shape_mismatch_count': len(
                        compatibility['shape_mismatches']
                    ),
                }
            )

            if compatibility['exact']:
                model.load_state_dict(variant, strict=True)
                print(
                    f'Loaded {model_name} state_dict from '
                    f'{candidate_path} ({variant_label}); '
                    f'{len(variant):,} tensors.'
                )
                return model

    diagnostic_frame = pd.DataFrame(diagnostics).sort_values(
        ['matched', 'missing_count', 'unexpected_count'],
        ascending=[False, True, True],
    )

    diagnostic_path = LOGS / (
        f'checkpoint_loading_diagnostic_'
        f'{model_name}_{checkpoint_path.stem}.csv'
    )
    diagnostic_frame.to_csv(diagnostic_path, index=False)

    best = diagnostic_frame.head(5).to_string(index=False)
    structure = '\n'.join(
        '  - ' + line
        for line in checkpoint_structure_summary(checkpoint_object)
    )

    raise RuntimeError(
        f'Checkpoint state_dict candidates were found, but none exactly '
        f'matched the reconstructed {model_name} architecture.\n'
        f'Checkpoint: {checkpoint_path}\n'
        f'Best compatibility candidates:\n{best}\n'
        f'Checkpoint structure:\n{structure}\n'
        f'Full diagnostic saved to: {diagnostic_path}'
    )

def load_model(model_name, seed):
    path = CHECKPOINTS[(model_name, seed)]
    print(f'Loading checkpoint: {path}')

    try:
        scripted_model = torch.jit.load(str(path), map_location='cpu')
        if isinstance(scripted_model, nn.Module):
            if nparams(scripted_model) != EXPECTED_PARAMS[model_name]:
                raise RuntimeError(
                    f'TorchScript parameter-count mismatch for {model_name}: '
                    f'{nparams(scripted_model):,} vs '
                    f'{EXPECTED_PARAMS[model_name]:,}'
                )
            return scripted_model.eval().to(DEVICE)
    except Exception:
        pass

    checkpoint_object = torch.load(
        path,
        map_location='cpu',
        weights_only=False,
    )

    full_model = full_model_from(checkpoint_object)
    if full_model is not None:
        if nparams(full_model) != EXPECTED_PARAMS[model_name]:
            raise RuntimeError(
                f'Serialized-model parameter-count mismatch for {model_name}: '
                f'{nparams(full_model):,} vs '
                f'{EXPECTED_PARAMS[model_name]:,}'
            )
        print(
            f'Loaded full serialized {model_name} model '
            f'from {path.name}.'
        )
        return full_model.eval().to(DEVICE)

    if not MODEL_NS:
        raise RuntimeError(
            'The checkpoint contains weights rather than a full model, '
            'but the exact Phase 2 model source input was not loaded.'
        )

    model = load_state_dict_recursively(
        model_name,
        checkpoint_object,
        path,
    )

    if nparams(model) != EXPECTED_PARAMS[model_name]:
        raise RuntimeError(
            f'Parameter-count mismatch for {model_name}: '
            f'{nparams(model):,} vs {EXPECTED_PARAMS[model_name]:,}'
        )

    return model.eval().to(DEVICE)


In [ ]:
class MRIDataset(Dataset):
    def __init__(self,df):self.df=df.reset_index(drop=True)
    def __len__(self):return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]
        with np.load(r.npz_path_resolved) as z: im=z['image'].astype(np.float32); ma=z['mask'].astype(np.uint8)
        if im.shape==(256,256,3):im=np.transpose(im,(2,0,1))
        return torch.from_numpy(np.ascontiguousarray(im)),torch.from_numpy(ma[None]),i

def loader(df):return DataLoader(MRIDataset(df),batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=USE_AMP,persistent_workers=NUM_WORKERS>0)

def logits_from(out):
    if torch.is_tensor(out):return out
    if isinstance(out,(list,tuple)):return logits_from(out[0])
    if isinstance(out,dict):
        for k in ['out','logits','mask','prediction','pred']:
            if k in out:return logits_from(out[k])
        return logits_from(next(iter(out.values())))
    raise TypeError(type(out))

def overlap(pred,target):
    pred,target=pred.bool(),target.bool(); dims=tuple(range(1,pred.ndim))
    tp=(pred&target).sum(dims).double(); fp=(pred&~target).sum(dims).double(); fn=(~pred&target).sum(dims).double(); tn=(~pred&~target).sum(dims).double()
    div=lambda a,b:torch.where(b>0,a/b,torch.ones_like(a))
    return {k:v.cpu().numpy() for k,v in {
      'tp':tp,'fp':fp,'fn':fn,'tn':tn,'dice':div(2*tp,2*tp+fp+fn),'iou':div(tp,tp+fp+fn),
      'precision':div(tp,tp+fp),'recall':div(tp,tp+fn),'lesion_pixels':target.sum(dims),'predicted_pixels':pred.sum(dims),
      'empty_prediction':(pred.sum(dims)==0).to(torch.uint8)}.items()}

def boundary(pred,target):
    if not SCIPY_AVAILABLE:return np.nan,np.nan
    pred,target=pred.astype(bool),target.astype(bool)
    if not pred.any() and not target.any():return 0.,0.
    if not pred.any() or not target.any():
        penalty=float(math.sqrt(pred.shape[0]**2+pred.shape[1]**2));return penalty,penalty
    ps=pred^binary_erosion(pred); ts=target^binary_erosion(target)
    d=np.concatenate([distance_transform_edt(~ts)[ps],distance_transform_edt(~ps)[ts]])
    return float(np.percentile(d,95)),float(d.mean())

In [ ]:
def evaluate(model_name,seed):
    out_csv=METRICS/f'crop_level_metrics_{model_name}_seed{seed}.csv'
    if out_csv.exists() and out_csv.stat().st_size>100:
        print('SKIP',out_csv.name);return out_csv
    threshold=FROZEN_THRESHOLDS[(model_name,seed)]; model=load_model(model_name,seed); rows=[]
    for split in SPLITS:
        df=manifest[manifest.split.astype(str).str.lower()==split].reset_index(drop=True)
        print(model_name,seed,split,len(df),'threshold',threshold)
        with torch.inference_mode():
            for images,masks,indices in loader(df):
                images,masks=images.to(DEVICE,non_blocking=True),masks.to(DEVICE,non_blocking=True)
                with torch.autocast(device_type=DEVICE.type,dtype=torch.float16,enabled=USE_AMP): logits=logits_from(model(images))
                if logits.ndim==3:logits=logits[:,None]
                if logits.shape[-2:]!=masks.shape[-2:]:logits=torch.nn.functional.interpolate(logits,size=masks.shape[-2:],mode='bilinear',align_corners=False)
                pred=torch.sigmoid(logits.float())>=threshold; met=overlap(pred,masks)
                pnp=pred[:,0].cpu().numpy(); mnp=masks[:,0].cpu().numpy()
                for j,idx in enumerate(indices.cpu().numpy()):
                    r=df.iloc[int(idx)]
                    rec={'sample_id':r.sample_id,'patient_id':str(r.patient_id),'case_id':str(r.case_id),'dataset':str(r.dataset).lower(),'split':split,'model':model_name,'seed':seed,'threshold':threshold,'npz_path':r.npz_path_resolved,
                         'crop_touches_border':bool(r.get('crop_touches_border',False)),'lesion_ratio_crop':r.get('lesion_ratio_crop',np.nan),'bbox_w':r.get('bbox_w',np.nan),'bbox_h':r.get('bbox_h',np.nan),'slice_index':r.get('slice_index',np.nan)}
                    for k,v in met.items():rec[k]=v[j].item() if hasattr(v[j],'item') else v[j]
                    rec['hd95_px'],rec['asd_px']=boundary(pnp[j],mnp[j]) if COMPUTE_BOUNDARY_METRICS and split in BOUNDARY_SPLITS else (np.nan,np.nan)
                    rows.append(rec)
    pd.DataFrame(rows).to_csv(out_csv,index=False)
    del model;gc.collect();
    if torch.cuda.is_available():torch.cuda.empty_cache()
    return out_csv

run_files=[evaluate(m,s) for m,s in RUNS]
print('Completed files:',len(run_files))

In [ ]:
crop=pd.concat([pd.read_csv(p) for p in sorted(METRICS.glob('crop_level_metrics_*_seed*.csv'))],ignore_index=True)
crop.to_csv(METRICS/'crop_level_metrics.csv',index=False)

gcols=['patient_id','dataset','split','model','seed','threshold']
patient=crop.groupby(gcols,as_index=False).agg(
 n_crops=('sample_id','size'),total_tp=('tp','sum'),total_fp=('fp','sum'),total_fn=('fn','sum'),total_tn=('tn','sum'),
 total_lesion_pixels=('lesion_pixels','sum'),total_predicted_pixels=('predicted_pixels','sum'),empty_prediction_count=('empty_prediction','sum'),
 mean_crop_dice=('dice','mean'),median_crop_dice=('dice','median'),mean_crop_iou=('iou','mean'),mean_crop_precision=('precision','mean'),mean_crop_recall=('recall','mean'),
 mean_crop_hd95_px=('hd95_px','mean'),mean_crop_asd_px=('asd_px','mean'),any_crop_touches_border=('crop_touches_border','max'),mean_lesion_ratio_crop=('lesion_ratio_crop','mean'))

def ratio(a,b):
    a,b=np.asarray(a,float),np.asarray(b,float);o=np.ones_like(a);np.divide(a,b,out=o,where=b>0);return o
patient['patient_dice']=ratio(2*patient.total_tp,2*patient.total_tp+patient.total_fp+patient.total_fn)
patient['patient_iou']=ratio(patient.total_tp,patient.total_tp+patient.total_fp+patient.total_fn)
patient['patient_precision']=ratio(patient.total_tp,patient.total_tp+patient.total_fp)
patient['patient_recall']=ratio(patient.total_tp,patient.total_tp+patient.total_fn)
patient.to_csv(METRICS/'patient_level_metrics.csv',index=False)

seedavg=patient.groupby(['patient_id','dataset','split','model'],as_index=False).agg(
 n_seeds=('seed','nunique'),n_crops=('n_crops','first'),patient_dice=('patient_dice','mean'),patient_dice_seed_sd=('patient_dice','std'),
 patient_iou=('patient_iou','mean'),patient_precision=('patient_precision','mean'),patient_recall=('patient_recall','mean'),mean_crop_dice=('mean_crop_dice','mean'),
 mean_crop_hd95_px=('mean_crop_hd95_px','mean'),mean_crop_asd_px=('mean_crop_asd_px','mean'),total_lesion_pixels=('total_lesion_pixels','mean'),
 any_crop_touches_border=('any_crop_touches_border','max'),mean_lesion_ratio_crop=('mean_lesion_ratio_crop','mean'),empty_prediction_count=('empty_prediction_count','mean'))
assert seedavg.n_seeds.eq(3).all()
seedavg.to_csv(METRICS/'patient_metrics_seed_averaged.csv',index=False)

summary=seedavg.groupby(['model','split'],as_index=False).agg(n_patients=('patient_id','nunique'),patient_dice_mean=('patient_dice','mean'),patient_dice_std=('patient_dice','std'),patient_dice_median=('patient_dice','median'),patient_iou_mean=('patient_iou','mean'),patient_precision_mean=('patient_precision','mean'),patient_recall_mean=('patient_recall','mean'))
summary.to_csv(METRICS/'patient_level_descriptive_summary.csv',index=False)
ranking=summary[summary.split=='validation'].sort_values('patient_dice_mean',ascending=False)
ranking.to_csv(LOGS/'validation_only_model_ranking.csv',index=False)
print(summary.to_string(index=False))

In [ ]:
report=['# Track B MRI — Final patient-level evaluation','','- No retraining.','- No threshold search.','- Thresholds frozen from validation.','- Test and Duke used only for confirmation.','- Patient Dice/IoU aggregate TP, FP and FN across all crops from the same patient.','- Boundary metrics, when enabled, are 2D resized-crop pixel distances, not physical 3D distances.','','## Validation-only ranking','',ranking.to_markdown(index=False),'','## Patient-level descriptive results','',summary.to_markdown(index=False)]
(WORK/'TRACKB_FINAL_PATIENT_LEVEL_RESULTS.md').write_text('\n'.join(report),encoding='utf-8')
zip_path=Path('/kaggle/working/TRACKB_FINAL_PATIENT_LEVEL_EVALUATION_LIGHT.zip')
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in WORK.rglob('*'):
        if p.is_file():z.write(p,p.relative_to(WORK.parent))
print('Final ZIP:',zip_path)

## Output for Notebook 2
